# Домашнє завдання 11 · LiDAR → RGB: перевірка дальності до об'єктів
**Заняття 11 · RGB + LiDAR / Radar · ≈ 3–4 години · 20 балів**

Збираємо конвеєр зі слайда «ДЗ: LiDAR → RGB для верифікації відстані» на **KITTI object**: хмару Velodyne проєктуємо
на кадр лівої кольорової камери, для кожного об'єкта з розмітки беремо точки в його боксі, рахуємо дальність з LiDAR
і з моно (`f · H / h`) і порівнюємо обидві з дальністю, яку дає 3D-бокс розмітки. Потім ламаємо калібрування і час
і дивимося, наскільки це псує результат. У дужках — назви слайдів.

| Частина | Що робимо | Слайди |
|---|---|---|
| **1 · калібрування і проєкція** | читаємо `calib/*.txt` → `P2 · R0_rect · Tr_velo_to_cam` → `project()` з тестом на синтетиці | «SE(3) і конвенція імен T_to_from», «Осі LiDAR і камери», «Конвенція KITTI» |
| **2 · overlay** | ≥ 5 кадрів, колір = глибина, колорбар | «Point-to-pixel: конвеєр і код», «LiDAR → RGB: колір = глибина» |
| **3 · GT дальності** | 3D-бокс з `label_2` → глибина його видимої поверхні | «Конвенція KITTI» (деталі, що ламають ДЗ) |
| **4 · дальність з LiDAR і з моно** | frustum боксу → перший щільний кластер → медіана; `f · H / h` | «Дальність до цілі: LiDAR vs моно vs стерео» |
| **5 · метрики** | таблиця по об'єктах, MAE / медіана / p90 по бінах 0–20 / 20–40 / 40+ м, відмови окремо | «ДЗ: LiDAR → RGB для верифікації відстані» |
| **6 · абляція** | помилка yaw ±0.5° / ±1° і розсинхрон Δt | «Як довести, що калібрування правильне», «Бюджет похибки синхронізації» |
| **7 · звіт** | чотири короткі висновки | — |

```
velodyne/*.bin ──► P2 · R0_rect · Tr_velo_to_cam ──► (u, v, z) ──► точки в боксі ──► кластер ──► d_lidar ─┐
image_2 + label_2: 2D-бокс, клас ───────────────────────────────────────────────► f · H / h ──► d_mono  ─┼─► |d − d_GT| по бінах
label_2: 3D-бокс ──► видима поверхня ──────────────────────────────────────────────────────► d_GT    ─┘
```

**Як працювати.** Читання файлів, малювання, GT і цикли по кадрах уже написані. Ваше — шість клітинок з `TODO`:
калібрування, проєкція, дальність з LiDAR, дальність з моно, метрики по бінах і збурення калібрування та часу. Після
кожного кроку `check()` показує ✅ або ❌. Відповіді пишете в клітинках «✍️».

In [ ]:
import sys
import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# спільний код заняття 11: читання KITTI object, розмітка, GT видимої поверхні, статуси відмов — той самий, що в LIVE DEMO 3
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'lessons' / '11-rgb-lidar-radar-fusion').is_dir())
sys.path.insert(0, str(REPO / 'lessons' / '11-rgb-lidar-radar-fusion'))
import kitti11 as K11

ROOT = K11.ROOT                   # artifacts/kitti/object/training або змінна KITTI_OBJECT_ROOT
N_FRAMES = 200                    # ≥ 20 кадрів за умовою; перші 200 за номером — у всіх ті самі кадри, у біні 40+ м ≈ 60 об'єктів

missing = [d for d in K11.PARTS if not (ROOT / d).is_dir()]
assert not missing, (f'у {ROOT} немає {missing}. Потрібен KITTI object training: velodyne, image_2, calib, label_2 '
                     '(python tools/fetch_kitti.py --object, див. README завдання) або змінна KITTI_OBJECT_ROOT.')
FRAMES = [p.stem for p in sorted((ROOT / 'label_2').glob('*.txt')) if K11.available(p.stem)][:N_FRAMES]
OUT_DIR = Path('outputs'); OUT_DIR.mkdir(exist_ok=True)
pd.set_option('display.precision', 2)
plt.rcParams['figure.dpi'] = 90
print(f'{ROOT}: {len(FRAMES)} кадрів ({FRAMES[0]} … {FRAMES[-1]})')


def check(ok, good, bad):
    print(('✅ ' + good) if ok else ('❌ ' + bad))


def load_points(stem):
    """velodyne/*.bin → N×3, м, у рамці Velodyne (x вперед, y ліворуч, z вгору). Масив лише для читання."""
    return K11.points_image(stem)[0]


def load_image(stem):
    """image_2/*.png → RGB у [0, 1]."""
    return K11.points_image(stem)[1]

# Частина 1 · Калібрування і проєкція

## 1.1 · Ланцюг KITTI (≈ 15 хв)

У `calib/000123.txt` кожен рядок — `ключ: числа`. Нам потрібні три матриці (слайд «Конвенція KITTI»):
- `Tr_velo_to_cam` (3×4) — з рамки Velodyne у рамку cam0. Попри назву «to», за нашою конвенцією це `T_cam0_velo`;
- `R0_rect` (3×3) — поворот ректифікації cam0;
- `P2` (3×4) — з ректифікованої рамки cam0 у пікселі лівої кольорової камери. Зсув cam0 → cam2 (≈ 6 см) уже сидить у `P2[:, 3]`.

Щоб їх перемножити, `R0_rect` і `Tr_velo_to_cam` доповнюємо до 4×4 одиничним рядком і стовпцем.

**TODO 1.1.** Допишіть `load_calib`: `R0` і `Tv` — матриці 4×4, `P = P2 · R0 · Tv` (3×4). Пастка зі слайда: без `R0_rect`
точки зсуваються на кілька пікселів і це легко не помітити на ближніх об'єктах.

In [ ]:
def load_calib(path):
    d = {}
    for ln in open(path):
        if ':' in ln:
            k, v = ln.split(':', 1)
            d[k.strip()] = np.array(v.split(), dtype=float)
    P2 = d['P2'].reshape(3, 4)
    R0 = Tv = P = ...   # TODO 1.1: R0 і Tv — 4×4, P = P2 · R0 · Tv (3×4)
    return dict(P2=P2, R0=R0, Tv=Tv, P=P)


# синтетичний calib з відомими матрицями: R0 — поворот на 1°, щоб пропущений R0_rect було видно
a = np.radians(1.0)
R0_t = np.array([[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]])
Tr_t = np.array([[0, -1, 0, 0.01], [0, 0, -1, -0.08], [1, 0, 0, -0.27]], float)
P2_t = np.array([[720, 0, 610, 44], [0, 720, 175, 0.2], [0, 0, 1, 0.003]], float)
test_path = OUT_DIR / '_calib_test.txt'
test_path.write_text('\n'.join(f'{k}: ' + ' '.join(map(str, m.ravel()))
                               for k, m in (('P2', P2_t), ('R0_rect', R0_t), ('Tr_velo_to_cam', Tr_t))) + '\n')
c_t = load_calib(test_path)
want = P2_t @ np.block([[R0_t, np.zeros((3, 1))], [np.zeros((1, 3)), 1]]) @ np.vstack([Tr_t, [0, 0, 0, 1]])
calib0 = load_calib(ROOT / 'calib' / f'{FRAMES[0]}.txt')
y = calib0['P'] @ np.array([10.0, 0, 0, 1])                  # точка 10 м прямо попереду Velodyne
check(np.shape(c_t['P']) == (3, 4) and np.allclose(c_t['P'], want) and 9 < y[2] < 11 and abs(y[0] / y[2] - calib0['P2'][0, 2]) < 60,
      f"ланцюг зібрано; кадр {FRAMES[0]}: точка 10 м попереду → u = {y[0] / y[2]:.0f} (cx = {calib0['P2'][0, 2]:.0f}), z = {y[2]:.2f} м",
      'P має бути 3×4 і дорівнювати P2 · R0 · Tv; точка 10 м попереду — z ≈ 10 м і u біля cx')

## 1.2 · Проєкція точок (≈ 20 хв)

Конвеєр зі слайда «Point-to-pixel»: однорідні координати → `P` → лишаємо точки **перед камерою** (`z > zmin`) → ділимо
на `z` → лишаємо ті, що потрапили в кадр. Без фільтра `z > 0` точки позаду камери після ділення на від'ємне `z`
з'являються в кадрі дзеркально.

**TODO 1.2.** `project(pts, P, W, H, zmin)` повертає `uv` (N×2, float), `z` (N) і булеву маску `m` (N): `z > zmin`
і `0 ≤ u < W`, `0 ≤ v < H`. Довжини всіх трьох — N, тобто як у вхідних точках; маску застосовує той, хто викликає.
Тест нижче перевіряє функцію на синтетиці з відомою відповіддю: заданий вручну pinhole, точки спереду, позаду й поза кадром.

In [ ]:
def project(pts, P, W, H, zmin=0.5):
    """pts: N×3 у рамці Velodyne → uv (N×2), z (N), m (N, bool): перед камерою і в кадрі."""
    uv = z = m = ...   # TODO 1.2: однорідні координати → P → z → ділення на z → маска
    return uv, z, m


# синтетика: камера з f = 700 px дивиться вперед, оптичний центр (600, 180), кадр 1200 × 360
K_s = np.array([[700, 0, 600], [0, 700, 180], [0, 0, 1.0]])
R_axes = np.array([[0, -1, 0], [0, 0, -1], [1, 0, 0.0]])        # FLU (Velodyne) → RDF (камера)
t_s = np.array([0.0, -0.08, -0.27])
P_s = K_s @ np.c_[R_axes, t_s]
rng = np.random.default_rng(0)
pts_s = np.c_[rng.uniform(-30, 60, 2000), rng.uniform(-25, 25, 2000), rng.uniform(-2, 3, 2000)]
Xc = pts_s @ R_axes.T + t_s
with np.errstate(divide='ignore', invalid='ignore'):
    u_want, v_want = 700 * Xc[:, 0] / Xc[:, 2] + 600, 700 * Xc[:, 1] / Xc[:, 2] + 180
m_want = (Xc[:, 2] > 0.5) & (u_want >= 0) & (u_want < 1200) & (v_want >= 0) & (v_want < 360)
uv_s, z_s, m_s = project(pts_s, P_s, 1200, 360)
uv10, z10, _ = project(np.array([[10.0, 0, 0]]), P_s, 1200, 360)
check(np.shape(uv_s) == (2000, 2) and np.array_equal(m_s, m_want) and np.allclose(uv_s[m_s], np.c_[u_want, v_want][m_want])
      and np.allclose(z_s[m_s], Xc[m_want, 2]) and abs(uv10[0, 0] - 600) < 1e-6 and abs(z10[0] - 9.73) < 1e-6,
      f'проєкція збігається з pinhole; у кадр потрапило {m_s.sum()} з 2000 точок, усі позаду камери відкинуто',
      'перевірте множення на P, ділення на z і маску (z > zmin і межі кадру)')

# Частина 2 · Overlay: колір = глибина (≈ 10 хв, код готовий)

Колір — логарифм глибини, як на слайді «LiDAR → RGB»: інакше всі точки далі 20 м зливаються в один колір. Збережіть
щонайменше 5 overlay. Подивіться на тонкі вертикалі (стовпи, краї машин) на різних дальностях: якщо точки «з'їжджають»
з краю однаково близько й далеко — це поворот, якщо лише близько — зсув (слайд «Як довести, що калібрування правильне»).
Висновок напишете в 7.1, після кропів у частині 5.

In [ ]:
DEPTH_NORM = plt.matplotlib.colors.LogNorm(vmin=3, vmax=80)


def overlay(stem, ax=None, P=None, s=1.5):
    img, pts = load_image(stem), load_points(stem)
    c = load_calib(ROOT / 'calib' / f'{stem}.txt')
    uv, z, m = project(pts, c['P'] if P is None else P, img.shape[1], img.shape[0])
    ax = ax or plt.gca()
    ax.imshow(img)
    sc = ax.scatter(uv[m, 0], uv[m, 1], c=z[m], s=s, cmap='turbo', norm=DEPTH_NORM, lw=0)
    ax.set_xlim(0, img.shape[1]); ax.set_ylim(img.shape[0], 0); ax.set_axis_off()
    return sc, m.mean()


OVERLAYS = FRAMES[:5]
for stem in OVERLAYS:
    fig, ax = plt.subplots(figsize=(14, 4.6))
    sc, share = overlay(stem, ax)
    ax.set_title(f'{stem}: у кадрі {share:.0%} точок хмари', loc='left')
    fig.colorbar(sc, ax=ax, fraction=0.025, pad=0.01, label='глибина, м (log)')
    fig.savefig(OUT_DIR / f'overlay_{stem}.png', bbox_inches='tight')
    plt.show()
check(all((OUT_DIR / f'overlay_{s}.png').exists() for s in OVERLAYS), f'збережено {len(OVERLAYS)} overlay в {OUT_DIR}/',
      'overlay не збережено')

# Частина 3 · GT дальності (≈ 10 хв, код готовий)

`label_2` дає для кожного об'єкта клас, прапорці `truncated` (0…1) і `occluded` (0 — видно повністю … 3 — невідомо),
2D-бокс у пікселях `image_2`, розміри `h, w, l` (м), `location` — **центр нижньої грані** 3D-боксу в ректифікованій
рамці cam0 — і поворот `ry` навколо вертикалі.

**Що вважати еталонною дальністю.** LiDAR бачить **поверхню**, повернуту до сенсора, а `location.z` — це центр боксу.
Для машини, яка стоїть до нас задом, різниця — половина довжини, ≈ 2 м, тобто в чотири рази більше за поріг 0.5 м з умови.
Тому за `d_GT` беремо медіанну глибину **видимої поверхні** 3D-боксу: через сітку пікселів його 2D-боксу пускаємо промені
з камери і для кожного беремо точку, де промінь входить у бокс. Сітка рівномірна в пікселях, як і точки LiDAR (вони рівномірні
за кутом), тож довгий бік машини, що в кадрі займає вузьку смужку, не перетягує медіану. Це та сама величина, яку міряє
кластер LiDAR, але порахована лише з розмітки. Для порівняння таблиця нижче показує й «наївні» варіанти: центр боксу і найближчу вершину.

In [ ]:
# код — у lessons/11-rgb-lidar-radar-fusion/kitti11.py, той самий, що в LIVE DEMO 3
from kitti11 import CLASS_H, labels as load_labels, box_corners, visible_depth

objs0 = load_labels(FRAMES[0])
print(pd.DataFrame([dict(cls=o['cls'], occl=o['occl'], trunc=o['trunc'],
                         d_GT=visible_depth(o, calib0['P2']), центр=o['loc'][2], вершина=box_corners(o)[:, 2].min())
                    for o in objs0]).to_string())

# Частина 4 · Дальність з LiDAR і з моно

## 4.1 · LiDAR: frustum → перший щільний кластер → медіана (≈ 40 мін)

У 2D-бокс потрапляють не лише точки об'єкта, а й земля під ним, фон за ним і все, що стоїть перед ним. Тому медіана
**всіх** точок у боксі — погана оцінка. Робимо так, як у LIVE DEMO 3:
1. беремо точки, чия проєкція в боксі (frustum). Бокс можна трохи звузити (`shrink` — частка з кожного боку), щоб
   на краях менше чіплялося фону;
2. якщо точок менше `min_pts` — дальності немає: повертаємо `(None, n)`. Це відмова, вона піде в таблицю окремим рядком;
3. гістограма глибин з кроком `bin_m`; перший від сенсора бін, у якому точок не менше `max(min_pts, 0.15 · max)`, —
   початок кластера. Поріг потрібен, щоб окремі точки пилу чи сусіднього об'єкта не стали «першим кластером»;
4. кластер — точки в межах `cluster_m` від початку біна; дальність — їхня медіана.

**TODO 4.1.** Функція повертає `(d, n)`: дальність у метрах або `None` і кількість точок у frustum.

In [ ]:
def lidar_range(uv, z, box, shrink=0.1, bin_m=0.5, min_pts=5, cluster_m=2.0):
    x0, y0, x1, y1 = box
    dx, dy = (x1 - x0) * shrink, (y1 - y0) * shrink
    # TODO 4.1: 1) точки в звуженому боксі  2) < min_pts → (None, n)
    #           3) гістограма, перший щільний бін  4) медіана кластера → (d, n)
    return None, 0


# синтетичний frustum: машина на 12 м (60 точок), земля 8–30 м, стіна на 35 м (300 точок), дві «пилинки» на 4 м
rng = np.random.default_rng(1)
z_p = np.r_[rng.normal(12.2, 0.15, 60), rng.uniform(8, 30, 40), rng.normal(35, 0.05, 300), [4.0, 4.1]]
uv_p = rng.uniform([110, 60], [190, 140], (len(z_p), 2))
d_p, n_p = lidar_range(uv_p, z_p, (100, 50, 200, 150), shrink=0.0)
d_few, n_few = lidar_range(uv_p[:4], z_p[:4], (100, 50, 200, 150), shrink=0.0)
check(d_p is not None and abs(d_p - 12.2) < 0.3 and n_p == len(z_p) and d_few is None and n_few == 4,
      f'машина на 12.2 м → {d_p:.2f} м з {n_p} точок; 4 точки → відмова',
      f'очікується ≈ 12.2 м (не стіна на 35 м і не пил на 4 м) і None для 4 точок; маємо {d_p}, {d_few}')

## 4.2 · Моно з відомим розміром (≈ 5 хв)

`Z = f · H / h`: `f` — фокусна відстань у пікселях (`P2[0, 0]`), `H` — **апріорна** висота класу з `CLASS_H` (не з
розмітки: на борту справжньої висоти немає), `h` — висота 2D-боксу в пікселях. Похибка — і від розміру конкретної машини,
і від ±2 px на краях боксу (слайд «Дальність до цілі»).

**TODO 4.2.** Одна формула.

In [ ]:
def mono_range(box, H_class, f):
    return ...   # TODO 4.2: f · H / h, h — висота боксу в пікселях


check(abs(mono_range(np.array([0, 100, 50, 135]), 1.75, 1000) - 50) < 1e-9,
      'людина 35 px при f = 1000 px → 50 м', 'людина висотою 35 px при f = 1000 px має бути на 50 м')

# Частина 5 · Таблиця і метрики

## Таблиця по об'єктах (≈ 5 хв, код готовий)

Кожен об'єкт отримує `status`. Метрики рахуємо лише по `ok`, а відмови показуємо окремо (критерій з умови):
- `оклюзія` — `occluded ≥ 2`: бокс закриває інший об'єкт, і перший кластер легко виявиться ним;
- `обрізаний` — `truncated > 0.3`: бокс урізаний краєм кадру, моно з такою висотою боксу бреше;
- `малий бокс` — висота < 25 px (поріг KITTI moderate);
- `< 5 точок` — LiDAR не дав дальності.

In [ ]:
def frame_rows(stem, P=None):
    c = load_calib(ROOT / 'calib' / f'{stem}.txt')
    img_h, img_w = load_image(stem).shape[:2]
    uv, z, m = project(load_points(stem), c['P'] if P is None else P, img_w, img_h)
    rows = []
    for o in load_labels(stem):
        d_l, n = lidar_range(uv[m], z[m], o['box'])
        h_px = o['box'][3] - o['box'][1]
        status = K11.status(o)
        status = '< 5 точок' if status == 'ok' and d_l is None else status
        rows.append(dict(frame=stem, cls=o['cls'], status=status, occl=o['occl'], trunc=o['trunc'], h_px=h_px, n_pts=n,
                         d_gt=visible_depth(o, c['P2']), d_center=o['loc'][2],
                         d_lidar=np.nan if d_l is None else d_l, d_mono=mono_range(o['box'], CLASS_H[o['cls']], c['P2'][0, 0])))
    return rows


df = pd.DataFrame([r for s in FRAMES for r in frame_rows(s)])
df['err_lidar'] = df.d_lidar - df.d_gt
df['err_mono'] = df.d_mono - df.d_gt
df['err_center'] = df.d_center - df.d_gt
ok = df[(df.status == 'ok') & df.d_gt.notna()]
df.to_csv(OUT_DIR / 'objects.csv', index=False)
print(f"об'єктів: {len(df)} у {df.frame.nunique()} кадрах, з них ok: {len(ok)}")
print(df.status.value_counts().rename('відмови і ok').to_string())
display(df.head(12))
check(df.frame.nunique() >= 20 and len(ok) >= 50, f"{df.frame.nunique()} кадрів, {len(ok)} об'єктів у метриках",
      "за умовою потрібно ≥ 20 кадрів і ≥ 50 об'єктів — збільште N_FRAMES")

## 5.1 · Метрики по бінах дальності (≈ 20 хв)

Біни — за еталонною дальністю `d_GT`: `0–20 м`, `20–40 м`, `40+ м`. У кожному біні — кількість об'єктів `N` і для
**модуля** похибки `MAE`, медіана і `p90`. `N` обов'язково: медіана з трьох об'єктів і з тридцяти важить по-різному.

**TODO 5.1.** `metrics_by_bin(err, d_gt)` повертає `DataFrame` з індексом `BINS` і стовпцями `N, MAE, median, p90`.
Порожній бін — `N = 0` і `NaN` у решті.

In [ ]:
BINS = {'0–20 м': (0, 20), '20–40 м': (20, 40), '40+ м': (40, np.inf)}


def metrics_by_bin(err, d_gt):
    err, d_gt = np.asarray(err, float), np.asarray(d_gt, float)
    ...   # TODO 5.1: для кожного біна — N і MAE / медіана / p90 модуля похибки


probe = metrics_by_bin([0.1, -0.3, 0.2, 2.0, -1.0], [5, 10, 15, 30, 35])
check(isinstance(probe, pd.DataFrame) and list(probe.index) == list(BINS) and probe.N.tolist() == [3, 2, 0]
      and np.isclose(probe.loc['0–20 м', 'MAE'], 0.2) and np.isclose(probe.loc['20–40 м', 'median'], 1.5)
      and np.isnan(probe.loc['40+ м', 'MAE']),
      'метрики по бінах рахуються правильно', 'очікуються N = [3, 2, 0], MAE 0–20 м = 0.2, медіана 20–40 м = 1.5, порожній бін — NaN')

In [ ]:
same = ok[ok.d_lidar.notna() & ok.d_mono.notna()]          # усі методи — на тих самих об'єктах
M = {'LiDAR': metrics_by_bin(same.err_lidar, same.d_gt), 'моно': metrics_by_bin(same.err_mono, same.d_gt),
     'центр боксу замість поверхні': metrics_by_bin(same.err_center, same.d_gt)}
display(pd.concat(M, axis=1))
med = M['LiDAR'].loc[['0–20 м', '20–40 м'], 'median']
check(bool((med.dropna() <= 0.5).all()), f'медіана |d_lidar − d_GT| до 40 м: {med.round(2).tolist()} м — критерій ≤ 0.5 м виконано',
      f'медіана до 40 м = {med.round(2).tolist()} м > 0.5 м: знайдіть причину й поясніть її у звіті (7.2)')

## Кропи на трьох дальностях (≈ 5 хв, код готовий)

Найближчий, середній і найдальший об'єкт з `ok`: бокс розмітки, точки LiDAR кольором глибини. Це матеріал для 7.1
(точки на краях об'єктів) і для частини 6.

In [ ]:
def crop(row, ax, P=None, pad=25):
    stem = row.frame
    sc, _ = overlay(stem, ax, P=P, s=6)
    o = next(o for o in load_labels(stem) if np.isclose(o['box'][3] - o['box'][1], row.h_px) and o['cls'] == row.cls)
    x0, y0, x1, y1 = o['box']
    ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec='w', lw=1.5))
    ax.set_xlim(x0 - pad, x1 + pad); ax.set_ylim(y1 + pad, y0 - pad)
    ax.set_title(f"{stem} {row.cls}: GT {row.d_gt:.1f} | LiDAR {row.d_lidar:.1f} | моно {row.d_mono:.1f} м", fontsize=9)
    return sc


picks = same.sort_values('d_gt').iloc[[0, len(same) // 2, -1]]
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for ax, (_, r) in zip(axs, picks.iterrows()):
    sc = crop(r, ax)
fig.colorbar(sc, ax=axs, fraction=0.015, label='глибина, м (log)')
fig.savefig(OUT_DIR / 'crops.png', bbox_inches='tight')
plt.show()

# Частина 6 · Абляція: калібрування і час

## 6.1 · Збурення `P` (≈ 30 хв)

Дві помилки, які дають «майже правильний» overlay:
- **yaw калібрування** — поворот на малий кут навколо вертикальної осі камери (вісь `y` рамки cam0, вниз). Множимо його
  між `R0` і `Tv`: `P' = P2 · R0 · R_y(yaw) · Tv`. На слайді «Як довести»: Δu ≈ f · δθ на будь-якій дальності;
- **розсинхрон Δt** — кадр знято на Δt пізніше за скан, а машина за цей час проїхала `v · Δt` вперед і повернула на
  `ω · Δt` ліворуч. Точку треба перевести в рамку Velodyne на момент кадру: `X' = R_z(ω·Δt)ᵀ · (X − [v·Δt, 0, 0])`,
  тобто `P' = P · M`, де `M` — це перетворення 4×4. Чистий поворот дає Δu ≈ f · ω · Δt (слайд «Бюджет похибки синхронізації»).

**TODO 6.1.** `perturb_P(c, yaw_deg, dt_s, v, w_deg)` повертає 3×4. При нульових збуреннях — рівно `c['P']`.

In [ ]:
def perturb_P(c, yaw_deg=0.0, dt_s=0.0, v=10.0, w_deg=30.0):
    """c — результат load_calib; v — швидкість вперед, м/с; w_deg — рискання ліворуч, °/с."""
    ...   # TODO 6.1: R_y(yaw) між R0 і Tv; M = [R_z(ω·Δt)ᵀ | −R_z(ω·Δt)ᵀ · (v·Δt, 0, 0)]


def u_of(P, X):
    y = P @ np.r_[X, 1.0]
    return y[0] / y[2], y[2]


f0 = calib0['P2'][0, 0]
X30 = np.array([30.0, 0, 0])
u0, z0 = u_of(calib0['P'], X30)
du_yaw = u_of(perturb_P(calib0, yaw_deg=1.0), X30)[0] - u0
du_w = u_of(perturb_P(calib0, dt_s=0.05, v=0.0, w_deg=30.0), X30)[0] - u0
dz_v = u_of(perturb_P(calib0, dt_s=0.1, v=10.0, w_deg=0.0), X30)[1] - z0
check(np.allclose(perturb_P(calib0), calib0['P']) and abs(abs(du_yaw) - f0 * np.radians(1.0)) < 0.1 * f0 * np.radians(1.0)
      and abs(abs(du_w) - f0 * np.radians(1.5)) < 0.1 * f0 * np.radians(1.5) and abs(dz_v + 1.0) < 0.05,
      f'yaw 1° → Δu = {du_yaw:+.1f} px (f·δθ = {f0 * np.radians(1):.1f}); 30°/с × 50 мс → {du_w:+.1f} px; '
      f'10 м/с × 100 мс → глибина {dz_v:+.2f} м',
      'без збурень має бути P; yaw 1° — |Δu| ≈ f·δθ; 30°/с × 50 мс — |Δu| ≈ f·1.5°; 10 м/с × 100 мс — точка на 1 м ближча')

## 6.2 · Як збурення псують дальність (≈ 5 хв, код готовий)

Ті самі `ok`-об'єкти, ті самі бокси; змінюється лише `P`. `> 1 м` — частка об'єктів, де дальність LiDAR відхилилась від
GT більше ніж на метр: кластер «з'їхав» на фон або землю. Δt — при 10 м/с і 30°/с, звичайний поворот машини на перехресті.

In [ ]:
CASES = [('без збурень', {}), ('yaw −1°', dict(yaw_deg=-1.0)), ('yaw −0.5°', dict(yaw_deg=-0.5)),
         ('yaw +0.5°', dict(yaw_deg=0.5)), ('yaw +1°', dict(yaw_deg=1.0)),
         ('Δt 10 мс', dict(dt_s=0.01)), ('Δt 50 мс', dict(dt_s=0.05)), ('Δt 100 мс', dict(dt_s=0.1))]
keys = set(zip(same.frame, same.cls, same.h_px))
abl = {}
for name, kw in CASES:
    rows = []
    for stem in FRAMES:
        c = load_calib(ROOT / 'calib' / f'{stem}.txt')
        rows += [r for r in frame_rows(stem, P=perturb_P(c, **kw)) if (r['frame'], r['cls'], r['h_px']) in keys]
    a = pd.DataFrame(rows)
    err = (a.d_lidar - a.d_gt).fillna(np.inf)               # дальність зникла — теж провал
    mb = metrics_by_bin(err.clip(upper=99), a.d_gt)
    abl[name] = {**{f'медіана {b}': mb.loc[b, 'median'] for b in BINS},
                 '> 1 м': float((err.abs() > 1).mean())}
ABL = pd.DataFrame(abl).T
display(ABL.style.format({**{f'медіана {b}': '{:.2f}' for b in BINS}, '> 1 м': '{:.0%}'}))

✍️ **Висновок 6.2** (2–3 речення, входить у бали звіту 7.3): у якому біні yaw ±1° шкодить найбільше і чому дальність
LiDAR узагалі псується, хоча самі точки не змінились? Порівняйте Δt 50 мс при 30°/с з yaw: який розсинхрон «коштує»
стільки ж, скільки 1° калібрування, і що це означає для бюджету з слайда «Бюджет похибки синхронізації»?

✍️ *Ваша відповідь:*

# Частина 7 · Звіт

Коротко, по 2–4 речення, з числами з таблиць вище.

✍️ **7.1 · Точки на краях.** Назвіть три кадри з об'єктами на різних дальностях (близько, 20–40 м, далеко), де видно,
що точки LiDAR лягають на краї об'єктів (overlay і кропи). Якщо десь «з'їжджають» — це схоже на поворот чи на зсув?

✍️ *Ваша відповідь:*

✍️ **7.2 · LiDAR проти моно.** Як змінюються MAE і p90 обох методів з дальністю і чому моно погіршується швидше?
Для якого класу моно помиляється найбільше (`same.groupby('cls')`)? Якщо медіана LiDAR до 40 м більша за 0.5 м —
поясніть, звідки похибка. І чому рядок «центр боксу замість поверхні» не можна використовувати як GT?

✍️ *Ваша відповідь:*

✍️ **7.3 · Абляція.** Висновок 6.2.

✍️ *Ваша відповідь:*

✍️ **7.4 · Три відмови.** Три конкретні об'єкти з `df[df.status != 'ok']` або з найбільшою `|err_lidar|` серед `ok`:
кадр, клас, що сталося і що з цим робити (прийом із заняття).

✍️ *Ваша відповідь:*

In [ ]:
report = dict(
    frames=len(FRAMES), objects=len(df), objects_ok=len(ok), objects_same=len(same),
    status=df.status.value_counts().to_dict(),
    metrics={k: v.reset_index(names='bin').to_dict('records') for k, v in M.items()},
    ablation=ABL.reset_index(names='case').to_dict('records'),
)
(OUT_DIR / 'report.json').write_text(json.dumps(report, ensure_ascii=False, indent=1, default=float))
print(f"→ {OUT_DIR / 'report.json'}")

## Бонус (за бажанням, не оцінюється)
- **Стерео.** Візьміть `image_3` і `P3` з того самого calib: база `B = (P2[0, 3] − P3[0, 3]) / f`. Порахуйте диспаритет
  `cv2.StereoSGBM` і `d_stereo = f · B / disparity` як медіану в боксі. На якій дальності стерео з базою 0.54 м програє
  LiDAR і чи виграє в моно (слайд «Дальність до цілі»: там база 12 см)?
- **z-buffer.** Замість першого щільного кластера зберіть для кожного пікселя найближчу точку (`np.minimum.at`) і розширте
  її на вікно 5×5. Чи зменшилась кількість об'єктів, де кластер «перескакує» на сусіда?
- **Справжній Δt.** У KITTI raw є `velodyne_points/timestamps_start.txt` / `_end.txt` і час кадрів: порівняйте реальний
  розсинхрон зі своїм бюджетом з частини 6.

# Здача
Усе здається **одним архівом** через особистий кабінет LMS. Порядок:
1. **Kernel → Restart Kernel and Run All Cells**: ноутбук має виконатись від початку до кінця, з усіма ✅
   (або з поясненням у 7.2, якщо не виконано критерій 0.5 м).
2. Заповніть відповіді в клітинках «✍️».
3. Збережіть ноутбук (**Ctrl+S**) і запустіть клітинку нижче: вона збере в архів `dz11_<прізвище>.zip` ноутбук,
   `outputs/report.json`, `outputs/objects.csv`, overlay і кропи.
4. Прикріпіть **цей один архів** в особистому кабінеті LMS. Окремі файли, посилання на диск чи репозиторій не приймаються.

In [ ]:
SURNAME = ''   # ✍️ прізвище латиницею, наприклад 'shevchenko'
NOTEBOOK = Path('05_lidar_rgb_range_student.ipynb')
FILES = [NOTEBOOK, OUT_DIR / 'report.json', OUT_DIR / 'objects.csv', OUT_DIR / 'crops.png',
         *[OUT_DIR / f'overlay_{s}.png' for s in OVERLAYS]]

assert SURNAME, 'впишіть прізвище в SURNAME'
missing = [str(f) for f in FILES if not f.exists()]
assert not missing, f'немає файлів {missing}: виконайте всі клітинки й збережіть ноутбук'
archive = Path(f'dz11_{SURNAME}.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in FILES:
        z.write(f, f.as_posix())
    names = z.namelist()
print(f'→ {archive.resolve()} ({archive.stat().st_size / 2**20:.1f} МБ): {", ".join(names)}')
print('Прикріпіть цей архів в особистому кабінеті LMS.')